# NB_01_Bronze_Ingest
Lands the raw CSV drops in Delta **exactly as received**: every column stays a string, nothing is
cleaned or dropped. Each row is stamped with the batch it came from, its source file and the load time.

Loading is incremental: `batch=<date>` is a partition folder, so filtering on batches newer than the
table's high-water mark means Spark only opens the new folders.

In [ ]:
landing_root = "Files/landing"
table_format = "delta"

In [ ]:
from pyspark.sql import functions as F

# keep batch=2026-01-31 as a string instead of letting Spark infer a date
spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")

ENTITIES = ["suppliers", "products", "customers", "fx_rates",
            "purchase_orders", "shipments", "invoice_lines", "bank_transactions"]

In [ ]:
summary = []
for entity in ENTITIES:
    table = f"bronze_{entity}"
    last_batch = None
    if spark.catalog.tableExists(table):
        last_batch = spark.table(table).agg(F.max("_batch")).first()[0]

    raw = (spark.read
           .option("header", True).option("inferSchema", False)
           .option("quote", '"').option("escape", '"')
           .csv(f"{landing_root}/{entity}")
           .withColumnRenamed("batch", "_batch"))
    if last_batch:
        raw = raw.filter(F.col("_batch") > last_batch)

    new = (raw
           .withColumn("_source_file", F.col("_metadata.file_path"))
           .withColumn("_ingested_at", F.current_timestamp()))

    new.write.format(table_format).mode("append").saveAsTable(table)
    loaded = spark.table(table).filter(F.col("_batch") > (last_batch or "")).count()
    summary.append((entity, last_batch, loaded))

spark.createDataFrame(summary, "entity string, previous_batch string, rows_loaded long").show(truncate=False)